# QENS Analysis Notebook App how to use

This notebook launches an interactive `ipywidgets` application for quasi-elastic neutron scattering (QENS) analysis of `.nxspe` files. It wraps the local `qens_library` package and provides two analysis routes:

1. **Least-squares Γ(Q) fitting** — extracts the half-width at half maximum, Γ(Q), then fits selected diffusion models such as Chudley–Elliott, Fickian, Singwi–Sjölander, or user-defined custom models.
2. **Bayesian MCMC forward modelling** — fits full forward models with `emcee`, if `emcee` is installed, and summarises posterior samples.

## Before running

- Keep the `qens_library/` folder in the same working directory as this notebook.
- Make sure your data files are in `.nxspe` format and are accessible from the notebook server.
- Run the notebook cells **from top to bottom**. The final cell displays the app.
- For Bayesian MCMC, `emcee` must be installed. If it is not installed, the least-squares workflow will still work.

## Basic workflow

1. **Run all cells** to initialise imports, plotting functions, widgets, and callbacks.
2. In **Data Source**, use the file browser to navigate to a folder containing `.nxspe` files.
3. Select one or more files and click **Add selected**, or use **Add all** to stage every `.nxspe` file in the current folder.
4. Choose the **primary file** from the staged list. The primary file is used for preview plots and representative residual plots.
5. Choose the fitting route:
   - **Least Squares Γ(Q)** for a fast model comparison on extracted Γ(Q).
   - **Bayesian MCMC** for posterior sampling of a selected forward model.
6. Adjust Q range, energy window, Q-bin count, and optional MCMC settings.
7. Click **Run Analysis**.
8. Review the generated tabs: parameter summary, S(Q,ω) map, fit/residuals, Γ(Q) comparison, posterior plots, and run log.

## Outputs

Each run creates a timestamped folder under `output/`, containing:

- `figures/` with rendered PNG plots.
- `hwhm.csv` containing extracted Γ(Q) values.
- `run_summary.json` containing selected settings, fitted parameters, and any Bayesian summary statistics.

## Custom Γ(Q) models

The **Custom Γ(Q) Model** panel lets you register simple Python expressions for Γ(Q). Use `q` for the Q array and define any fitted parameters in the parameter table. Available symbols include `np`, `HBAR_MEV_PS`, `ce_hwhm`, `fickian_hwhm`, and `ss_hwhm`.

Example formulas:

```python
HBAR_MEV_PS * D * q**2
HBAR_MEV_PS / tau * (1 - np.sinc(q*l/np.pi))
```

After registering a model, tick it in the model selection panel before running the analysis.

## Troubleshooting

- **No files staged**: add at least one `.nxspe` file in the Data Source panel.
- **No primary file set**: select a primary file from the dropdown after staging files.
- **MCMC unavailable**: install `emcee` or use least-squares fitting.
- **Unexpected fitting errors**: narrow the Q range, check the energy window, or inspect whether the selected `.nxspe` files are valid.

## 1. Imports, package setup, and plotting defaults

This cell imports the numerical, plotting, widget, and local QENS-library dependencies used throughout the app. It also defines colour/label dictionaries and Matplotlib defaults so that all plots generated later have consistent styling.

In [1]:
# Notebook app initialisation.
# This cell prepares the Python path, imports scientific dependencies,
# imports the local qens package, and configures global plotting defaults.

import os, sys, json, warnings, datetime, traceback, pathlib, io as _io
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.join(os.getcwd(), "qens_library"))

import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, LinearSegmentedColormap
from scipy.optimize import curve_fit, nnls
from scipy.signal import fftconvolve
from scipy.ndimage import gaussian_filter
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

import qens
from qens                   import Config
from qens.constants         import HBAR_MEV_PS
from qens.io                import load_dataset, read_nxspe, inspect_nxspe
from qens.preprocessing     import fit_elastic_peak, assign_resolution
from qens.models            import (
    lorentz, gnorm,
    fickian_hwhm, ce_hwhm, ss_hwhm,
    rot_widths_isotropic, rot_widths_anisotropic, bessel_weights,
    predict_sqw, register_model, get_model, available_models,
)
from qens.fitting           import (
    extract_hwhm, save_hwhm_csv,
    build_data_bins, build_resolution_bins, find_map,
)
from qens.sampling          import run_mcmc, summarise_samples

try:
    import emcee
    _EMCEE = True
except ImportError:
    _EMCEE = False



_MODEL_COLORS = {
    "ce":                "#c0392b",
    "fickian":           "#2471a3",
    "ss":                "#1e8449",
    "lorentz":           "#6c3483",
    "translation_only":  "#2471a3",
    "isotropic_rotor":   "#1e8449",
    "anisotropic_rotor": "#c0392b",
}
_MODEL_LABELS = {
    "ce":                "Chudley-Elliott",
    "fickian":           "Fickian",
    "ss":                "Singwi-Sjolander",
    "lorentz":           "Lorentzian (mean HWHM)",
    "translation_only":  "Translation-only forward model",
    "isotropic_rotor":   "Isotropic rotor forward model",
    "anisotropic_rotor": "Anisotropic rotor forward model",
}

_CMAP = LinearSegmentedColormap.from_list(
    "qens",
    ["#0a0e1a","#0c2d6b","#1565c0","#42a5f5","#e3f2fd","#ff8f00","#e65100"],
    N=512,
)

plt.rcParams.update({
    "font.family":     "DejaVu Sans",
    "axes.linewidth":  0.8,
    "axes.edgecolor":  "#555",
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "legend.framealpha": 0.92,
    "legend.edgecolor":  "#ccc",
    "figure.dpi":        100,
})


print(f"qens {qens.__version__}   emcee={_EMCEE}")
print(f"available forward models: {available_models()}")
print("Initialised.")


qens 2.0.0   emcee=True
available forward models: ['anisotropic_rotor', 'isotropic_rotor', 'translation_only']
Initialised.


## 2. Plotting helpers and model-fitting utilities

These helper functions do not display the app directly. They produce the figures used in the output tabs and implement the least-squares Γ(Q) fits for both built-in and custom models.

In [2]:
# Figure functions and least-squares Γ(Q) fitter (uses qens v1.0.0 API).

_custom_models = {}        # legacy registry kept for the "custom Γ(Q) model" widget


def _fit_gamma_ls(q_arr, g_arr, g_err, model_name):
    """Least-squares fit of Γ(Q) to one of the bundled translational models.
    Returns dict of best-fit parameters or {"error": "..."} on failure."""
    try:
        if model_name == "ce":
            p, _ = curve_fit(ce_hwhm, q_arr, g_arr, sigma=g_err,
                             p0=[0.30, 2.5], bounds=([1e-3, 0.5], [3, 6]),
                             maxfev=8000)
            return {"D": float(p[0]), "l": float(p[1]),
                    "tau": float(p[1]**2 / (6 * p[0]))}
        if model_name == "fickian":
            p, _ = curve_fit(fickian_hwhm, q_arr, g_arr, sigma=g_err,
                             p0=[0.30], bounds=([1e-3], [3]), maxfev=8000)
            return {"D": float(p[0])}
        if model_name == "ss":
            p, _ = curve_fit(ss_hwhm, q_arr, g_arr, sigma=g_err,
                             p0=[0.30, 1.0], bounds=([1e-3, 1e-2], [3, 20]),
                             maxfev=8000)
            return {"D": float(p[0]), "tau_s": float(p[1])}
        if model_name in _custom_models:
            info = _custom_models[model_name]
            func = info["func"]
            p0_v = [pd[1] for pd in info["param_defs"]]
            lo_v = [pd[2] for pd in info["param_defs"]]
            hi_v = [pd[3] for pd in info["param_defs"]]
            p, _ = curve_fit(func, q_arr, g_arr, sigma=g_err,
                             p0=p0_v, bounds=(lo_v, hi_v), maxfev=8000)
            return {pd[0]: float(p[i]) for i, pd in enumerate(info["param_defs"])}
        # phenomenological "lorentz" — just the mean Γ
        return {"mean_hwhm_mev": float(np.mean(g_arr))}
    except Exception as exc:
        return {"error": str(exc)}


def _despine(ax):
    """Remove top/right axes spines for cleaner notebook figures."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


#  S(Q,ω) heat map

def fig_sqw_map(d_inc, ewin=1.0):
    """Create an S(Q, ω) intensity heat map for the selected energy window.

    Parameters
    ----------
    d_inc : dict
        Incoherent data dictionary returned by the qens loading/preprocessing steps.
    ewin : float
        Symmetric energy-transfer window, in meV, shown around zero energy.
    """
    good = d_inc["good"]
    q = d_inc["q"]
    e = d_inc["e"]
    emask = (e >= -ewin) & (e <= ewin)
    qs = np.argsort(q[good])
    img = d_inc["data"][np.ix_(good, emask)]
    img = np.where(np.isfinite(img) & (img > 0), img, np.nan)
    ism = gaussian_filter(np.where(np.isfinite(img[qs]), img[qs], 0), sigma=[1.5, 0.8])
    ism[ism <= 0] = np.nan
    vmin, vmax = np.nanpercentile(ism, 2), np.nanpercentile(ism, 99)

    fig, ax = plt.subplots(figsize=(6.5, 4.8))
    im = ax.pcolormesh(e[emask], q[good][qs], ism, cmap=_CMAP,
                       norm=LogNorm(vmin=max(vmin, 1e-6), vmax=vmax),
                       shading="auto", rasterized=True)
    ax.axvline(0, color="#888", lw=0.8, ls="--")
    ax.set_xlabel(r"Energy transfer  $\hbar\omega$  (meV)", fontsize=11)
    ax.set_ylabel(r"Momentum transfer  $Q$  ($\AA^{-1}$)", fontsize=11)
    ax.set_title(r"$S(Q,\omega)$  measured intensity", fontsize=11, pad=8)
    cb = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.038)
    cb.set_label(r"$S(Q,\omega)$  (a.u.)", fontsize=9)
    cb.ax.tick_params(labelsize=8)
    ax.tick_params(labelsize=9)
    fig.tight_layout()
    return fig


#  single-spectrum fit + residual
def fig_fit_residuals(d_inc, ce_D, ce_l, q_target=1.06):
    """Plot the measured spectrum, Chudley-Elliott fit, and residual near a target Q value."""
    """Show data + CE fit at one Q with residuals. Uses ce_hwhm() to get the
    HWHM, then convolves a Lorentzian with the Gaussian resolution kernel."""
    gp = d_inc["good"]
    qg = d_inc["q"][gp]
    sr = d_inc["sigma_res"]
    emask = (d_inc["e"] >= -0.8) & (d_inc["e"] <= 0.8)
    ew = d_inc["e"][emask]

    near = np.where(np.abs(qg - q_target) < 0.10)[0]
    if len(near) == 0:
        near = np.argsort(np.abs(qg - q_target))[:4]
    spec = np.nanmean([d_inc["data"][gp[j]][emask] for j in near], axis=0)
    errs = np.sqrt(np.nanmean([d_inc["errs"][gp[j]][emask]**2 for j in near], axis=0))
    spec = np.where(np.isfinite(spec), spec, 0)
    errs = np.where(errs > 0, errs, spec.max() * 0.05)
    sn = spec / spec.max()
    en = errs / spec.max()

    wf = np.linspace(-0.8, 0.8, 1000)
    dt = wf[1] - wf[0]
    gamma = float(ce_hwhm(np.asarray([q_target]), ce_D, ce_l)[0])

    el = gnorm(wf, sr); el /= el.max()
    ql_ = fftconvolve(lorentz(wf, gamma), gnorm(wf, sr), mode="same") * dt
    ql = ql_ / ql_.max() if ql_.max() > 0 else ql_
    sn_fine = np.interp(wf, ew, sn)
    amp, _ = nnls(np.column_stack([el, ql, np.ones(len(wf))]), sn_fine)
    fit = amp[0] * el + amp[1] * ql + amp[2]
    fit_d = np.interp(ew, wf, fit)
    resid = (sn - fit_d) / en
    chi2r = float(np.sum(resid**2) / max(len(resid) - 4, 1))

    fig, axes = plt.subplots(2, 1, figsize=(6.8, 6.0),
                             gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
    axes[0].errorbar(ew, sn, yerr=en, fmt=".", color="#333", ms=3.5,
                     elinewidth=0.7, alpha=0.8, label="Data", zorder=5)
    axes[0].fill_between(wf, amp[2], amp[0]*el + amp[2],
                         alpha=0.22, color="#1565c0", label="Elastic")
    axes[0].fill_between(wf, amp[2], amp[1]*ql + amp[2],
                         alpha=0.22, color="#e65100", label="Quasi-elastic")
    axes[0].plot(wf, fit, "-", color="#c0392b", lw=2.0,
                 label=rf"CE model  $\chi^2_r={chi2r:.2f}$")
    axes[0].set_ylabel(r"$S(Q,\omega)$  normalised", fontsize=11)
    axes[0].set_title(rf"Single-spectrum fit  $Q={q_target:.2f}$ $\AA^{{-1}}$",
                      fontsize=11, pad=8)
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.18, lw=0.6)
    _despine(axes[0])

    axes[1].axhline(0, color="#555", lw=0.9)
    axes[1].axhline( 2, color="#999", ls="--", lw=0.7)
    axes[1].axhline(-2, color="#999", ls="--", lw=0.7)
    axes[1].plot(ew, resid, ".", color="#c0392b", ms=3.5, alpha=0.85)
    axes[1].set_xlabel(r"Energy transfer  $\hbar\omega$  (meV)", fontsize=11)
    axes[1].set_ylabel(r"Residual ($\sigma$)", fontsize=10)
    axes[1].set_ylim(-4.8, 4.8)
    axes[1].grid(True, alpha=0.18, lw=0.6)
    _despine(axes[1])
    fig.tight_layout(h_pad=0.4)
    return fig, chi2r


#  Γ(Q) model comparison
def fig_hwhm_comparison(q_hwhm, g_hwhm, g_err, model_results,
                        samples_map=None, res_hwhm_uev=50):
    q2d = q_hwhm**2
    q_f = np.linspace(max(q_hwhm.min() * 0.85, 0.05), q_hwhm.max() * 1.10, 350)
    q2f = q_f**2

    fig, ax = plt.subplots(figsize=(7.2, 5.2))

    # CE-only posterior fan if Bayesian samples are present with (D, l) chains
    smp_ce = (samples_map or {}).get("ce_legacy")
    if smp_ce is not None and smp_ce.shape[1] >= 2:
        d_s, l_s = smp_ce[:, 0], np.abs(smp_ce[:, 1])
        idx = np.random.default_rng(0).choice(len(d_s), min(300, len(d_s)), replace=False)
        fan = np.array([ce_hwhm(q_f, d_s[i], l_s[i]) * 1000 for i in idx])
        ax.fill_between(q2f, np.percentile(fan, 2.5, axis=0),
                        np.percentile(fan, 97.5, axis=0),
                        alpha=0.14, color=_MODEL_COLORS["ce"])

    for model, ls_res in model_results.items():
        col = _MODEL_COLORS.get(model, "#333")
        lbl = _MODEL_LABELS.get(model, model)
        if "error" in ls_res:
            continue

        D = ls_res.get("D", 0.30)
        L = ls_res.get("l", 2.50)
        ts = ls_res.get("tau_s", 1.0)
        c2 = ls_res.get("_chi2r")

        if model == "ce":
            tag = rf"$D={D:.4f}$, $\ell={L:.4f}$"
            y = ce_hwhm(q_f, D, L) * 1000
        elif model == "fickian":
            tag = rf"$D={D:.4f}$"
            y = fickian_hwhm(q_f, D) * 1000
        elif model == "ss":
            tag = rf"$D={D:.4f}$, $\tau_s={ts:.3f}$"
            y = ss_hwhm(q_f, D, ts) * 1000
        elif model in _custom_models:
            info  = _custom_models[model]
            func  = info["func"]
            pvals = [ls_res.get(pd[0], pd[1]) for pd in info["param_defs"]]
            y = func(q_f, *pvals) * 1000
            tag = "  ".join(f"{pd[0]}={ls_res.get(pd[0], pd[1]):.4f}" for pd in info["param_defs"])
        else:
            mh = ls_res.get("mean_hwhm_mev", g_hwhm.mean()) * 1000
            ax.axhline(mh, color=col, lw=1.8, ls="-.",
                       label=rf"{lbl}  $\bar{{\Gamma}}={mh:.0f}$ $\mu$eV")
            continue

        suffix = rf"  ($\chi^2_r={c2:.3f}$)" if c2 is not None else ""
        ax.plot(q2f, y, "-", color=col, lw=2.0, zorder=5,
                label=f"{lbl}  {tag}{suffix}")

    ax.errorbar(q2d, g_hwhm * 1000, yerr=2 * g_err * 1000, fmt="o",
                color="#111", ms=5.5, capsize=3.5, elinewidth=1.4,
                label=r"Data  $\pm 2\sigma$", zorder=6)
    ax.axhline(res_hwhm_uev, color="#888", ls=":", lw=1.3,
               label=rf"Resolution HWHM  $={res_hwhm_uev:.0f}$ $\mu$eV")
    ax.axhspan(0, res_hwhm_uev * 1.15, alpha=0.04, color="#888")
    ax.set_xlabel(r"$Q^2$  ($\AA^{-2}$)", fontsize=11)
    ax.set_ylabel(r"$\Gamma(Q)$  ($\mu$eV)", fontsize=11)
    ax.set_title(r"$\Gamma(Q)$ vs $Q^2$"
                 + ("  —  model comparison" if len(model_results) > 1 else ""),
                 fontsize=11, pad=8)
    ax.legend(fontsize=8.5, loc="upper left")
    ax.set_xlim(left=-0.04)
    ax.set_ylim(bottom=-8)
    ax.grid(True, alpha=0.18, lw=0.6)
    _despine(ax)
    fig.tight_layout()
    return fig


#  forward-model posterior panel
def fig_forward_posteriors(samples, model_name):
    """Plot marginal posterior histograms for Bayesian forward-model samples."""
    """Histograms of every parameter of a forward model (uses qens registry to
    get parameter names). Adds the anisotropy ratio if applicable."""
    fm = get_model(model_name)
    names = list(fm.param_names)
    cols = [samples[:, i] for i in range(samples.shape[1])]
    if model_name == "anisotropic_rotor" and samples.shape[1] >= 4:
        names.append("D_s/D_t")
        cols.append(samples[:, 3] / np.maximum(samples[:, 2], 1e-12))

    n = len(cols)
    cols_per_row = min(4, n)
    rows = (n + cols_per_row - 1) // cols_per_row
    fig, axes = plt.subplots(rows, cols_per_row, figsize=(3.6 * cols_per_row, 3.0 * rows),
                             squeeze=False)
    axes = axes.flatten()

    palette = ["#c0392b", "#7f8c8d", "#1e8449", "#2471a3", "#8e44ad", "#e67e22"]
    for ax, name, arr, col in zip(axes, names, cols, palette * (1 + n // len(palette))):
        med = np.median(arr)
        lo, hi = np.percentile(arr, [2.5, 97.5])
        cnt, _, _ = ax.hist(arr, bins=50, density=True, color=col,
                            alpha=0.78, edgecolor="white", lw=0.3)
        pk = cnt.max() if len(cnt) else 1.0
        ax.axvspan(lo, hi, alpha=0.16, color=col)
        ax.axvline(med, color="#111", lw=1.6, label=rf"median$={med:.4f}$")
        ax.text((lo + hi) / 2, pk * 1.10, rf"95% [{lo:.4f}, {hi:.4f}]",
                ha="center", fontsize=7.5, color=col)
        ax.set_xlabel(name, fontsize=10)
        ax.set_ylabel("density", fontsize=9)
        ax.set_title(name, fontsize=10, color=col, fontweight="bold")
        ax.set_ylim(0, pk * 1.30)
        ax.legend(fontsize=7.5)
        ax.grid(True, alpha=0.18, lw=0.6)
        _despine(ax)

    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(f"Posterior distributions  —  {_MODEL_LABELS.get(model_name, model_name)}",
                 fontsize=11, fontweight="bold")
    fig.tight_layout()
    return fig


#  preview pane (used before run)
def fig_data_preview(d):
    """Create a quick preview figure for the selected primary data file."""
    good = d["good"]
    q_arr = d["q"][good]
    e = d["e"]
    ei = d["ei"]

    ewin = min(ei * 0.45, 1.2)
    emask = (e >= -ewin) & (e <= ewin)
    qs = np.argsort(q_arr)
    img = d["data"][np.ix_(good, emask)]
    img = np.where(np.isfinite(img) & (img > 0), img, np.nan)
    ism = gaussian_filter(np.where(np.isfinite(img[qs]), img[qs], 0.0), sigma=[1.5, 0.8])
    ism[ism <= 0] = np.nan
    vmin = max(np.nanpercentile(ism, 2), 1e-8)
    vmax = np.nanpercentile(ism, 99)

    n_lo = max(3, len(good) // 3)
    avg = np.nanmean([d["data"][good[j]] for j in range(n_lo)], axis=0)
    avg = np.where(np.isfinite(avg), avg, 0.0)
    emask2 = (e >= -ewin * 0.6) & (e <= ewin * 0.6)
    emask_el = (e >= -0.15) & (e <= 0.15)
    peak_heights = np.array([d["data"][i][emask_el].max() if emask_el.any() else 0.0 for i in good])
    peak_heights = np.where(np.isfinite(peak_heights), peak_heights, 0.0)

    fig = plt.figure(figsize=(13, 5.0))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 1], height_ratios=[5, 1],
                          hspace=0.08, wspace=0.32)
    ax_map   = fig.add_subplot(gs[:, 0])
    ax_peak  = fig.add_subplot(gs[0, 1])
    ax_strip = fig.add_subplot(gs[1, 1])

    im = ax_map.pcolormesh(e[emask], q_arr[qs], ism, cmap=_CMAP,
                           norm=LogNorm(vmin=vmin, vmax=vmax),
                           shading="auto", rasterized=True)
    ax_map.axvline(0.0, color="#aaa", lw=0.8, ls="--", alpha=0.6)
    ax_map.set_xlabel(r"$\hbar\omega$  (meV)", fontsize=11)
    ax_map.set_ylabel(r"$Q$  (Å$^{-1}$)", fontsize=11)
    ax_map.set_title(rf"$S(Q,\omega)$  —  {d['name']}", fontsize=10, pad=7)
    cb = fig.colorbar(im, ax=ax_map, pad=0.02, fraction=0.038)
    cb.set_label("intensity", fontsize=8)
    cb.ax.tick_params(labelsize=7)
    ax_map.tick_params(labelsize=9)
    _despine(ax_map)

    norm_pk = avg[emask2].max() if avg[emask2].max() > 0 else 1.0
    ax_peak.fill_between(e[emask2], 0, avg[emask2] / norm_pk, alpha=0.35, color="#2471a3")
    ax_peak.plot(e[emask2], avg[emask2] / norm_pk, color="#2471a3", lw=1.8)
    ax_peak.axvline(0.0, color="#aaa", lw=0.8, ls="--", alpha=0.6)
    ax_peak.set_ylabel("normalised intensity", fontsize=9)
    ax_peak.set_title(f"Elastic peak  (lowest-Q detectors)  "
                      f"FWHM ≈ {d.get('fwhm_res', 0)*1000:.0f} µeV  [{d.get('res_source', '—')}]",
                      fontsize=8.5, pad=6)
    ax_peak.tick_params(labelsize=8)
    ax_peak.set_xlim(e[emask2][0], e[emask2][-1])
    ax_peak.set_ylim(bottom=-0.04)
    ax_peak.grid(True, alpha=0.18, lw=0.6)
    _despine(ax_peak)
    plt.setp(ax_peak.get_xticklabels(), visible=False)

    strip = peak_heights[np.argsort(q_arr)][np.newaxis, :]
    ax_strip.imshow(strip, aspect="auto", cmap="plasma",
                    extent=[q_arr.min(), q_arr.max(), 0, 1],
                    vmin=0, vmax=max(peak_heights.max(), 1e-8))
    ax_strip.set_xlabel(r"$Q$  (Å$^{-1}$)", fontsize=9)
    ax_strip.set_yticks([])
    ax_strip.set_title("detector peak intensity  (dark = weak / masked)", fontsize=8, pad=4)
    ax_strip.tick_params(labelsize=8)

    fig.tight_layout(pad=1.2)
    return fig


print("Backend functions loaded.")


Backend functions loaded.


## 3. App styling

The app uses a small CSS block to make the widget interface easier to read. This affects only the notebook display and does not change the scientific analysis.

In [3]:
_CSS = """
<style>
.qs-app * { box-sizing: border-box; font-family: "Helvetica Neue", Arial, sans-serif; }
.qs-heading {
    font-size: .74em; font-weight: 700; letter-spacing: .10em;
    text-transform: uppercase; color: #4a5568;
    border-bottom: 1px solid #e2e8f0; padding-bottom: 5px; margin-bottom: 10px;
}
.qs-lbl    { font-size: .85em; color: #1a202c; margin: 6px 0 3px 0; font-weight: 600; }
.qs-info   { color: #1e3a5f; background:#e0f2fe; border-left:3px solid #0284c7; padding:6px 10px; border-radius:3px; font-size:.85em;}
.qs-warn   { color: #7c2d12; background:#fff7ed; border-left:3px solid #ea580c; padding:6px 10px; border-radius:3px; font-size:.85em;}
.qs-err    { color: #7f1d1d; background:#fef2f2; border-left:3px solid #dc2626; padding:6px 10px; border-radius:3px; font-size:.85em;}
.qs-ok     { color: #14532d; background:#f0fdf4; border-left:3px solid #16a34a; padding:6px 10px; border-radius:3px; font-size:.85em;}
.qs-tbl { border-collapse: collapse; width: 100%; font-size:.86em; }
.qs-tbl th { background:#f1f5f9; color:#1e293b; text-align:left; padding:5px 8px; border-bottom:1px solid #cbd5e1;}
.qs-tbl td { padding:4px 8px; border-bottom:1px solid #f1f5f9; }
.qs-tbl tr.qs-section td { background:#e0f2fe; padding:5px 8px; color:#0c4a6e; }
.qs-section td { font-size:.82em; letter-spacing:.06em; text-transform:uppercase; }
.qs-log-ok    { color: #166534; }
.qs-log-warn  { color: #c2410c; }
.qs-log-err   { color: #b91c1c; }
.qs-log-ts    { color: #94a3b8; font-size:.80em; margin-right:6px; }
.qs-path {
    font-family: "SFMono-Regular", "Consolas", monospace;
    font-size: .79em; color: #1e3a5f; background: #f1f5f9;
    border: 1px solid #cbd5e1; border-radius: 4px;
    padding: 4px 10px; margin-bottom: 6px; word-break: break-all;
}
.qs-badge {
    display: inline-block; background: #1e3a5f; color: #fff;
    font-size: .75em; padding: 1px 9px; border-radius: 12px;
    vertical-align: middle; margin-left: 6px;
}
.qs-cm-formula {
    font-family: "SFMono-Regular", "Consolas", monospace;
    font-size: .82em; background: #f8fafc;
    border: 1px solid #cbd5e1; border-radius: 4px;
    padding: 6px 8px;
}
.qs-cm-tag {
    display: inline-block; background: #78350f; color: #fef3c7;
    font-size: .73em; padding: 2px 8px; border-radius: 10px;
    margin: 2px 4px 2px 0; font-family: monospace;
}
</style>
"""


# Custom Γ(Q) models — registered through the UI.
# These are evaluated against the *least-squares Γ(Q) vs Q²* fit only;
# the Bayesian forward-model path uses the qens registry directly.
def register_custom_model(key, label, formula_str, param_defs, color="#78350f"):
    """Register a Γ(Q) model from a Python expression string.

    The expression has access to: ``q``, ``np``, ``HBAR_MEV_PS``,
    ``ce_hwhm``, ``fickian_hwhm``, ``ss_hwhm`` and any parameters declared
    in ``param_defs`` (a list of (name, p0, lo, hi) tuples).
    """
    import re
    key = re.sub(r"[^a-z0-9_]", "_", key.lower().strip())
    if not key:
        raise ValueError("Model key cannot be empty.")
    ns = {
        "np":           np,
        "HBAR_MEV_PS":  HBAR_MEV_PS,
        "hbar_mevps":   HBAR_MEV_PS,         # alias for users typing the old name
        "ce_hwhm":      ce_hwhm,
        "fickian_hwhm": fickian_hwhm,
        "ss_hwhm":      ss_hwhm,
        "ce":           ce_hwhm,             # legacy aliases
        "fickian":      fickian_hwhm,
        "ss_model":     ss_hwhm,
    }
    args = ", ".join(["q"] + [pd[0] for pd in param_defs])
    func = eval(f"lambda {args}: {formula_str}", ns)
    test_q = np.array([0.5, 1.0, 1.5])
    test_args = [test_q] + [pd[1] for pd in param_defs]
    result = func(*test_args)
    if not np.all(np.isfinite(result)):
        raise ValueError("Formula returned non-finite values.")
    _custom_models[key] = {
        "func": func, "param_defs": param_defs, "color": color,
        "label": label, "formula": formula_str,
    }
    _MODEL_COLORS[key] = color
    _MODEL_LABELS[key] = label
    return True


def unregister_custom_model(key):
    _custom_models.pop(key, None)
    _MODEL_COLORS.pop(key, None)
    _MODEL_LABELS.pop(key, None)


print("CSS and custom-model registry loaded.")


CSS and custom-model registry loaded.


## 4. Custom Γ(Q) model registration widgets

This section builds the controls that let users define additional Γ(Q) models from Python expressions. Registered models are added to the model-selection panel and can be fitted by the same least-squares routine as the bundled models.

In [4]:
# Widgets and callbacks for defining custom Γ(Q) models.
# The controls collect a model key, display label, formula, colour, and bounded parameters.

_w_cm_key = widgets.Text(value="", placeholder="e.g. my_model (no spaces)", layout=widgets.Layout(width="100%"))

_w_cm_label = widgets.Text(value="", placeholder="e.g. My Diffusion Model", layout=widgets.Layout(width="100%"))
_w_cm_formula = widgets.Textarea(
    value="", placeholder="Python expression for Gamma(Q) in meV.\nAvailable: q (array), np, HBAR_MEV_PS, ce_hwhm, fickian_hwhm, ss_hwhm, and your parameter names.\nExamples:\n  HBAR_MEV_PS * D * q**2\n  HBAR_MEV_PS / tau * (1 - np.sinc(q*l/np.pi))",
    layout=widgets.Layout(width="100%", height="110px"))
_w_cm_color = widgets.Dropdown(
    options=[("Amber","#92400e"),("Teal","#0f766e"),("Indigo","#3730a3"),("Rose","#9f1239"),
             ("Cyan","#0e7490"),("Slate","#334155"),("Lime","#3f6212"),("Orange","#c2410c")],
    value="#92400e", layout=widgets.Layout(width="100%"))




_cm_param_rows = []
_cm_params_vbox = widgets.VBox([], layout=widgets.Layout(margin="4px 0"))
_cm_params_status = widgets.HTML("")




def _make_param_row():
    """Create one editable parameter row for the custom-model form."""
    w_name = widgets.Text(placeholder="name (e.g. D)", layout=widgets.Layout(width="90px"))
    w_p0 = widgets.FloatText(value=0.30, layout=widgets.Layout(width="80px"))
    w_lo = widgets.FloatText(value=0.0, layout=widgets.Layout(width="80px"))
    w_hi = widgets.FloatText(value=10.0, layout=widgets.Layout(width="80px"))
    w_rm = widgets.Button(description="Remove", button_style="danger", layout=widgets.Layout(width="74px", height="28px"))
    row_dict = dict(name=w_name, p0=w_p0, lo=w_lo, hi=w_hi, rm=w_rm)
    row_widget = widgets.HBox([w_name, w_p0, w_lo, w_hi, w_rm],
                              layout=widgets.Layout(align_items="center", margin="2px 0"))
    def _remove(_):
        if row_dict in _cm_param_rows:
            _cm_param_rows.remove(row_dict)
            _refresh_params_vbox()
    w_rm.on_click(_remove)
    return row_widget, row_dict



def _refresh_params_vbox():
    """Redraw the parameter rows shown in the custom-model form."""
    if not _cm_param_rows:
        _cm_params_vbox.children = [widgets.HTML('<span style="color:#94a3b8;font-size:.82em">No parameters added yet.</span>')]
        return
    header = widgets.HBox([
        widgets.HTML('<span style="color:#64748b;font-size:.78em;width:90px;display:inline-block">Name</span>'),
        widgets.HTML('<span style="color:#64748b;font-size:.78em;width:80px;display:inline-block;margin-left:4px">p0</span>'),
        widgets.HTML('<span style="color:#64748b;font-size:.78em;width:80px;display:inline-block;margin-left:4px">Lower</span>'),
        widgets.HTML('<span style="color:#64748b;font-size:.78em;width:80px;display:inline-block;margin-left:4px">Upper</span>')
    ], layout=widgets.Layout(margin="0 0 2px 0"))
    rows = [header] + [rd["_widget"] for rd in _cm_param_rows]
    _cm_params_vbox.children = rows




def _add_param_row(_=None):
    """Add a blank parameter row to the custom-model form."""
    row_widget, row_dict = _make_param_row()
    row_dict["_widget"] = row_widget
    _cm_param_rows.append(row_dict)
    _refresh_params_vbox()

_btn_add_param = widgets.Button(description="Add parameter", layout=widgets.Layout(width="130px", height="28px"))
_btn_add_param.on_click(_add_param_row)

_btn_register = widgets.Button(description="Register Model", button_style="primary",
                               layout=widgets.Layout(width="150px", height="34px"))
_w_register_status = widgets.HTML("")
_w_registered_list = widgets.VBox([widgets.HTML('<span style="color:#94a3b8;font-size:.82em">No custom models registered yet.</span>')])




def _rebuild_registered_list():
    """Update the visible list of custom models currently registered."""
    if not _custom_models:
        _w_registered_list.children = [widgets.HTML('<span style="color:#94a3b8;font-size:.82em">No custom models registered yet.</span>')]
        return
    rows = []
    for key, info in _custom_models.items():
        col = info["color"]
        label = info["label"]
        swatch = f'<span style="display:inline-block;width:12px;height:12px;background:{col};border-radius:2px;margin-right:6px;vertical-align:middle"></span>'
        rm_btn = widgets.Button(description="Remove", button_style="danger", layout=widgets.Layout(width="74px", height="24px"))
        name_html = widgets.HTML(f'{swatch}<strong style="font-size:.85em">{label}</strong><code style="font-size:.76em;color:#64748b;margin-left:8px">key: {key}</code>')
        row = widgets.HBox([name_html, rm_btn], layout=widgets.Layout(align_items="center", margin="2px 0"))
        def _make_remover(k):
            def _remove(_):
                unregister_custom_model(k)
                if k in w_models:
                    del w_models[k]
                    _refresh_model_panel()
                _rebuild_registered_list()
            return _remove
        rm_btn.on_click(_make_remover(key))
        rows.append(row)
    _w_registered_list.children = rows




def _refresh_model_panel():
    """Synchronise the model-selection panel with built-in and custom models."""
    rows = []
    for key, _name, _desc, _default in _MODEL_DEFS:
        if key in w_models:
            cb = w_models[key]
        else:
            continue
        col = _MODEL_COLORS.get(key, "#334155")
        rows.append(widgets.HBox([cb, widgets.HTML(f'<span style="color:#718096;font-size:.82em;margin-left:4px">— {_desc}</span>')],
                                 layout=widgets.Layout(align_items="center", margin="2px 0")))
    for key, info in _custom_models.items():
        if key not in w_models:
            col = info["color"]
            cb = widgets.Checkbox(value=True, description=info["label"], indent=False,
                                  layout=widgets.Layout(width="auto", margin="0"),
                                  style={"description_width":"0px","description_color":col})
            w_models[key] = cb
        else:
            cb = w_models[key]
        col = info["color"]
        rows.append(widgets.HBox([cb, widgets.HTML(f'<span style="color:#718096;font-size:.82em;margin-left:4px">— custom</span>')],
                                 layout=widgets.Layout(align_items="center", margin="2px 0")))
    w_model_box.children = rows




def _on_register(_):
    """Validate custom-model inputs and register the model with the app."""
    _w_register_status.value = ""
    key = _w_cm_key.value.strip()
    label = _w_cm_label.value.strip() or key
    formula = _w_cm_formula.value.strip()
    color = _w_cm_color.value
    if not key:
        _w_register_status.value = '<div class="qs-err">Model key is required.</div>'
        return
    if not formula:
        _w_register_status.value = '<div class="qs-err">Formula is required.</div>'
        return
    param_defs = []
    for rd in _cm_param_rows:
        name = rd["name"].value.strip()
        if name:
            param_defs.append((name, rd["p0"].value, rd["lo"].value, rd["hi"].value))
    try:
        register_custom_model(key, label, formula, param_defs, color=color)
        _w_register_status.value = f'<div class="qs-ok">Model "{label}" registered as <code>{key}</code>.</div>'
        _rebuild_registered_list()
        _refresh_model_panel()
        _w_cm_key.value = ""
        _w_cm_label.value = ""
        _w_cm_formula.value = ""
        _cm_param_rows.clear()
        _refresh_params_vbox()
    except Exception as exc:
        _w_register_status.value = f'<div class="qs-err"><strong>Registration failed:</strong> {exc}</div>'



_btn_register.on_click(_on_register)
_refresh_params_vbox()




def _lbl_cm(text):
    """Return a styled label for the custom-model UI."""
    return widgets.HTML(f'<span class="qs-lbl">{text}</span>')




_cm_form = widgets.VBox([
    _lbl_cm("Model key  (short, no spaces)"), _w_cm_key,
    _lbl_cm("Display label"), _w_cm_label,
    _lbl_cm("Gamma(Q) formula  (meV) — uses q as the Q array"), _w_cm_formula,
    widgets.HTML('<div style="font-size:.78em;color:#64748b;margin:3px 0 8px 0">Available names: &nbsp;<code>q</code> &nbsp;<code>np</code> &nbsp;<code>HBAR_MEV_PS</code> &nbsp;<code>ce_hwhm</code> &nbsp;<code>fickian_hwhm</code> &nbsp;<code>ss_hwhm</code> + your parameter names below</div>'),
    _lbl_cm("Colour"), _w_cm_color,
    _lbl_cm("Parameters"), _cm_params_vbox, _btn_add_param,
    widgets.HBox([_btn_register], layout=widgets.Layout(margin="10px 0 4px 0")),
    _w_register_status,
    _lbl_cm("Registered custom models"), _w_registered_list,
])




print("Custom model widgets ready.")

Custom model widgets ready.


## 5. File browser widget

The `FileBrowser` class provides an in-notebook browser for `.nxspe` files. It handles folder navigation, staging multiple files for analysis, choosing a primary file, and exposing the selected paths to the pipeline.

In [5]:
class FileBrowser:
    """Small ipywidgets file browser specialised for staging `.nxspe` inputs.

    The browser keeps track of three things required by the pipeline:
    the current folder, the list of staged data files, and the primary file used
    for representative previews/diagnostic plots.
    """
    EXT = ".nxspe"

    def __init__(self, start_path="."):
        """Build the visible browser controls and attach navigation callbacks."""
        self._cwd = pathlib.Path(start_path).resolve()
        self._staged = {}

        self._w_path = widgets.HTML()
        self._btn_up = widgets.Button(description="Up", layout=widgets.Layout(width="56px", height="30px"))
        self._w_goto = widgets.Text(placeholder="Paste path and press Enter", continuous_update=False,
                                    layout=widgets.Layout(flex="1", height="30px"))
        self._btn_go = widgets.Button(description="Go", button_style="primary",
                                      layout=widgets.Layout(width="46px", height="30px"))
        self._nav = widgets.HBox([self._btn_up, self._w_goto, self._btn_go],
                                 layout=widgets.Layout(align_items="center", margin="0 0 6px 0"))

        self._w_dirs = widgets.Select(options=[], rows=4, layout=widgets.Layout(width="100%"))
        self._w_files = widgets.SelectMultiple(options=[], rows=6, layout=widgets.Layout(width="100%"))

        self._btn_add = widgets.Button(description="Add selected", button_style="success",
                                       layout=widgets.Layout(width="110px", height="28px"))
        self._btn_add_all = widgets.Button(description="Add all", layout=widgets.Layout(width="80px", height="28px"))
        self._btn_clr = widgets.Button(description="Clear all", button_style="warning",
                                       layout=widgets.Layout(width="90px", height="28px"))
        self._act = widgets.HBox([self._btn_add, self._btn_add_all, self._btn_clr],
                                 layout=widgets.Layout(margin="4px 0"))

        self._w_staged = widgets.SelectMultiple(options=[], rows=4, layout=widgets.Layout(width="100%"))
        self._btn_rm = widgets.Button(description="Remove selected", button_style="danger",
                                      layout=widgets.Layout(width="130px", height="26px"))
        self._rm_row = widgets.HBox([self._btn_rm], layout=widgets.Layout(margin="3px 0 0 0"))

        self._w_primary = widgets.Dropdown(options=[], description="", layout=widgets.Layout(width="100%"))
        self._w_status = widgets.HTML()





        def _lbl(text):
            return widgets.HTML(f'<span class="qs-lbl">{text}</span>')

        self.widget = widgets.VBox([
            self._w_path, self._nav,
            _lbl("Folders"), self._w_dirs,
            _lbl('Files  <span style="font-weight:400;color:#718096">(Ctrl-click / Shift-click for multi-select)</span>'),
            self._w_files, self._act,
            _lbl("Staged files"), self._w_staged, self._rm_row,
            _lbl('Primary file  <span style="font-weight:400;color:#718096">(INC reference file for spectrum plot)</span>'),
            self._w_primary, self._w_status,
        ])

        self._btn_up.on_click(self._go_up)
        self._btn_go.on_click(self._go_to)
        self._w_goto.observe(self._go_to, names="value")
        self._w_dirs.observe(self._on_dir_click, names="value")
        self._btn_add.on_click(self._add_highlighted)
        self._btn_add_all.on_click(self._add_all)
        self._btn_clr.on_click(lambda _: self._clear_all())
        self._btn_rm.on_click(self._remove_highlighted)
        self._w_primary.observe(self._on_primary, names="value")
        self._refresh()





    def _list(self, path):
        dirs, files = [], []
        try:
            entries = sorted(path.iterdir(), key=lambda e: (not e.is_dir(), e.name.lower()))
        except PermissionError:
            return [], []
        for e in entries:
            if e.name.startswith("."): continue
            if e.is_dir():
                try:
                    n = sum(1 for x in e.iterdir() if x.suffix == self.EXT)
                except PermissionError:
                    n = 0
                cnt = f"  [{n}]" if n else ""
                dirs.append((f"{e.name}/{cnt}", str(e)))
            elif e.suffix == self.EXT:
                kb = e.stat().st_size / 1024
                files.append((f"{e.name}  ({kb:.0f} KB)" if kb>0 else e.name, str(e)))
        return dirs, files




    def _refresh(self):
        """Refresh folder/file listings and staged-file dropdowns."""
        dirs, files = self._list(self._cwd)
        self._w_path.value = f'<div class="qs-path">{self._cwd}</div>'
        self._w_dirs.options = dirs or [("(no sub-folders)", "")]
        self._w_files.options = files or [("(no .nxspe files here)", "")]
        self._refresh_staged()




    def _refresh_staged(self):
        staged = list(self._staged)
        names = [pathlib.Path(p).name for p in staged]
        self._w_staged.options = list(zip(names, staged)) if staged else [("(none staged)", "")]
        dd = [(pathlib.Path(p).name, p) for p in staged]
        prev = self._w_primary.value
        self._w_primary.options = dd if dd else [("—", "")]
        if prev and prev in staged:
            self._w_primary.value = prev
        elif staged:
            inc = [p for p in staged if "inc" in pathlib.Path(p).name.lower()]
            self._w_primary.value = inc[0] if inc else staged[0]
        n = len(staged)
        self._w_status.value = f'<span class="qs-badge">{n} file{"s" if n!=1 else ""} staged</span>' if n else '<span style="color:#94a3b8;font-size:.82em">No files staged</span>'




    def _go_up(self, _=None):
        p = self._cwd.parent
        if p != self._cwd:
            self._cwd = p
            self._refresh()




    def _go_to(self, _=None):
        raw = self._w_goto.value.strip()
        if not raw: return
        p = pathlib.Path(raw).expanduser().resolve()
        if p.is_dir():
            self._cwd = p
            self._w_goto.value = ""
            self._refresh()
        else:
            self._w_status.value = f'<span class="qs-warn">Path not found: {raw}</span>'



    def _on_dir_click(self, change):
        val = change.get("new", "")
        if val and pathlib.Path(val).is_dir():
            self._cwd = pathlib.Path(val)
            self._refresh()


    def _add_highlighted(self, _=None):
        for lbl, val in (self._w_files.options or []):
            if val in (self._w_files.value or ()) and val:
                self._staged[val] = True
        self._refresh_staged()

   
    def _add_all(self, _=None):
        for lbl, val in (self._w_files.options or []):
            if val and pathlib.Path(val).suffix == self.EXT:
                self._staged[val] = True
        self._refresh_staged()


    def _remove_highlighted(self, _=None):
        for val in (self._w_staged.value or ()):
            self._staged.pop(val, None)
        self._refresh_staged()


    def _clear_all(self):
        self._staged.clear()
        self._refresh_staged()


    def _on_primary(self, change):
        cb = getattr(self, '_preview_cb', None)
        if cb is not None and change.get('new'):
            cb()


    @property
    def selected_files(self):
        return list(self._staged)


    @property
    def primary(self):
        v = self._w_primary.value
        return v if (v and v not in ("", "—")) else (self.selected_files[0] if self.selected_files else None)


_browser = FileBrowser()

## 6. Analysis controls

This section defines the user-facing controls for model choice, least-squares versus Bayesian workflow, MCMC settings, Q/energy filtering, binning, and output tabs.

In [6]:
# User-facing controls for selecting models, method, Q/energy windows, and MCMC settings.
# These widgets store the analysis choices read later by run_pipeline().

#  least-squares Γ(Q) models the user can tick on/off
_MODEL_DEFS = [
    ("ce",       "Chudley-Elliott",   "jump diffusion on a lattice",        True),
    ("fickian",  "Fickian",           "continuous Brownian motion",          True),
    ("ss",       "Singwi-Sjolander",  "correlated jump diffusion",           False),
    ("lorentz",  "Lorentzian",        "phenomenological single Lorentzian",  False),
]

w_models = {}
_model_rows = []
for _key, _name, _desc, _default in _MODEL_DEFS:
    _col = _MODEL_COLORS[_key]
    cb = widgets.Checkbox(value=_default, description=_name, indent=False,
                          layout=widgets.Layout(width="auto", margin="0"),
                          style={"description_width": "0px",
                                 "description_color": _col})
    w_models[_key] = cb
    _model_rows.append(widgets.HBox(
        [cb, widgets.HTML(f'<span style="color:#718096;font-size:.82em;margin-left:4px">— {_desc}</span>')],
        layout=widgets.Layout(align_items="center", margin="2px 0")))
w_model_box = widgets.VBox(_model_rows)
w_model_warn = widgets.HTML("")


def _chk_models(change):
    """Warn the user if all Γ(Q) model checkboxes are disabled."""
    any_on = any(cb.value for cb in w_models.values())
    w_model_warn.value = ('<div class="qs-warn" style="margin-top:6px">'
                          'Select at least one Γ(Q) model.</div>'
                          if not any_on else "")
for _cb in w_models.values():
    _cb.observe(_chk_models, names="value")


#  Bayesian forward-model selector
w_bayes_model = widgets.Dropdown(
    options=[
        ("Translation only (D*, ⟨u²⟩)",          "translation_only"),
        ("Isotropic rotor (D*, ⟨u²⟩, D_r)",      "isotropic_rotor"),
        ("Anisotropic rotor (D*, ⟨u²⟩, D_t, D_s)","anisotropic_rotor"),
    ],
    value="anisotropic_rotor",
    style={"description_width": "0px"},
    layout=widgets.Layout(width="100%"))


w_method = widgets.ToggleButtons(
    options=[("Least Squares Γ(Q)", "ls"),
             ("Bayesian MCMC",      "bayes")],
    value="ls",
    style={"button_width": "180px", "font_weight": "600"})


#  MCMC settings (visible only when Bayesian selected)
_sl = dict(continuous_update=False, style={"description_width": "72px"},
           layout=widgets.Layout(width="100%"))
w_nwalkers = widgets.IntSlider(value=32, min=8, max=128, step=8,
                               description="Walkers", **_sl)
w_nwarmup  = widgets.IntSlider(value=300, min=50, max=2000, step=50,
                               description="Burn-in", **_sl)
w_nkeep    = widgets.IntSlider(value=1000, min=100, max=5000, step=100,
                               description="Samples", **_sl)
w_mcmc_box = widgets.VBox([
    widgets.HTML('<span class="qs-lbl" style="margin-top:8px">Bayesian forward model</span>'),
    w_bayes_model,
    widgets.HTML('<span class="qs-lbl" style="margin-top:8px">MCMC settings</span>'),
    w_nwalkers, w_nwarmup, w_nkeep,
], layout=widgets.Layout(display="none"))


def _toggle_mcmc(change):
    """Show or hide MCMC controls depending on the selected fitting method."""
    w_mcmc_box.layout.display = "flex" if change["new"] == "bayes" else "none"
w_method.observe(_toggle_mcmc, names="value")


#  range / window / binning
_rs = dict(continuous_update=False, readout_format=".2f",
           style={"description_width": "70px"},
           layout=widgets.Layout(width="100%"))
w_qrange   = widgets.FloatRangeSlider(value=[0.30, 2.50], min=0.0, max=4.0,
                                      step=0.05, description="Q (1/A)", **_rs)
w_ewin     = widgets.FloatSlider(value=0.80, min=0.20, max=2.0, step=0.05,
                                 description="±E (meV)", **_rs)
w_nbins    = widgets.IntSlider(value=13, min=4, max=30, step=1,
                               description="Q bins",
                               continuous_update=False,
                               style={"description_width": "70px"},
                               layout=widgets.Layout(width="100%"))
w_q_target = widgets.BoundedFloatText(value=1.06, min=0.1, max=3.5, step=0.05,
                                      description="Q target",
                                      layout=widgets.Layout(width="180px"),
                                      style={"description_width": "68px"})


#  run button + progress
w_run = widgets.Button(description="Run Analysis", button_style="primary",
                       layout=widgets.Layout(width="180px", height="40px"))
w_prog = widgets.IntProgress(value=0, min=0, max=100, bar_style="info",
                             layout=widgets.Layout(width="100%", height="10px",
                                                   display="none"))
w_prog_lbl = widgets.HTML("")


#  output panes
out_params = widgets.Output()
out_map    = widgets.Output()
out_fit    = widgets.Output()
out_hwhm   = widgets.Output()
out_post   = widgets.Output()
out_log    = widgets.Output()


print("Widgets created.")


Widgets created.


## 7. Interface layout

These small layout helpers assemble the individual controls into card-like panels and combine them into the final app object displayed at the end of the notebook.

In [7]:
def _card(heading, *children):
    """Wrap controls in a consistently styled card with a heading."""
    return widgets.VBox(
        [widgets.HTML(f'<div class="qs-heading">{heading}</div>')] + list(children),
        layout=widgets.Layout(border="1px solid #e2e8f0", border_radius="6px",
                              padding="12px 14px", margin="0 0 10px 0", background_color="#ffffff"))



def _lbl(txt, mt="0"):
    """Return a styled label used in the main control panels."""
    return widgets.HTML(f'<span class="qs-lbl" style="margin-top:{mt}">{txt}</span>')



panel_data = _card("Data Source", _browser.widget)
panel_models = _card("Physical Models", w_model_box, w_model_warn)
panel_custom = _card("Custom Model", _cm_form)
panel_fitting = _card("Fitting Method", w_method, w_mcmc_box)
panel_params = _card("Analysis Parameters",
    _lbl("Q range (\u00c5\u207b\u00b9)"), w_qrange,
    _lbl("Energy window", mt="6px"), w_ewin,
    _lbl("Q bins", mt="6px"), w_nbins,
    _lbl("Single-spectrum Q target (\u00c5\u207b\u00b9)", mt="6px"), w_q_target)



panel_run = widgets.VBox([
    w_run,
    widgets.HBox([w_prog, w_prog_lbl], layout=widgets.Layout(align_items="center", margin="8px 0 0 0"))
], layout=widgets.Layout(padding="14px 16px", border="1.5px solid #2563eb",
                         border_radius="6px", background_color="#eff6ff"))



left_col = widgets.VBox(
    [panel_data, panel_models, panel_custom, panel_fitting, panel_params, panel_run],
    layout=widgets.Layout(width="370px", min_width="350px", margin="0 14px 0 0"))



tabs = widgets.Tab(children=[out_params, out_map, out_fit, out_hwhm, out_post, out_log])
_tab_titles = ["Parameters", "S(Q,w) Map", "Fit / Residuals", "Gamma vs Q2", "Posteriors", "Run Log"]
for i, t in enumerate(_tab_titles):
    tabs.set_title(i, t)



right_col = widgets.VBox([tabs], layout=widgets.Layout(flex="1", min_width="0"))



_header = widgets.HTML(
    '<div style="border-left:4px solid #2563eb;padding:10px 16px;background:#f8fafc;margin-bottom:12px">'
    '<span style="font-size:1.15em;font-weight:700;color:#1e3a5f">QENS Analysis Studio</span>'
    '<span style="font-size:.85em;color:#64748b;margin-left:14px">Select files — choose models — run analysis</span>'
    '</div>')



app = widgets.VBox(
    [_header, widgets.HBox([left_col, right_col], layout=widgets.Layout(width="100%", align_items="flex-start"))],
    layout=widgets.Layout(width="100%"))



print("Layout assembled.")

Layout assembled.


## 8. Pipeline execution, rendering, and saving outputs

`run_pipeline` is the main callback triggered by **Run Analysis**. It validates the selected files/settings, loads the data, preprocesses resolution information, extracts Γ(Q), runs the selected fitting workflow, renders figures, and writes run outputs to disk.

In [8]:
# Pipeline callbacks and output helpers.
# The Run Analysis button is connected to run_pipeline(), which coordinates validation,
# loading, preprocessing, fitting, plotting, and saving.

def _prog(pct, msg=""):
    """Update the progress bar and status message."""
    w_prog.value = int(pct)
    w_prog_lbl.value = (f'<span style="font-size:.82em;color:#64748b;'
                        f'margin-left:6px">{msg}</span>')


def _log(msg, kind=""):
    """Append a timestamped message to the run log panel."""
    cls = {"ok": "qs-log-ok", "warn": "qs-log-warn", "err": "qs-log-err"}.get(kind, "")
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    with out_log:
        display(HTML(f'<span class="qs-log-ts">[{ts}]</span>'
                     f'<span class="{cls}">{msg}</span><br>'))


def _render(out_widget, fig, save_path=None):
    """Render a Matplotlib figure into an output widget and optionally save it."""
    buf = _io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=140)
    if save_path:
        fig.savefig(save_path, bbox_inches="tight", dpi=200)
    plt.close(fig)
    buf.seek(0)
    img = widgets.Image(value=buf.read(), format="png",
                        layout=widgets.Layout(width="100%", max_width="820px"))
    with out_widget:
        clear_output(wait=True)
        display(img)


def run_pipeline(_btn):
    """Run the complete QENS analysis workflow using the current widget settings.

    The function is intentionally linear so notebook users can follow each stage:
    validate settings, load files, preprocess the primary dataset, extract Γ(Q),
    fit selected models, render figures, save CSV/JSON outputs, and restore the UI.
    """
    for o in [out_params, out_map, out_fit, out_hwhm, out_post, out_log]:
        with o: clear_output(wait=True)
    w_run.disabled = True
    w_run.description = "Running..."
    w_prog.layout.display = "flex"
    w_prog.bar_style = "info"
    _prog(0, "Starting")

    sel_models = [k for k, cb in w_models.items() if cb.value]
    if not sel_models:
        with out_params:
            display(HTML('<div class="qs-warn">Select at least one Γ(Q) model.</div>'))
        w_run.disabled = False
        w_run.description = "Run Analysis"
        return

    method = w_method.value
    bayes_model = w_bayes_model.value
    q_lo, q_hi = w_qrange.value
    ewin = w_ewin.value
    nbins = w_nbins.value
    q_tgt = w_q_target.value

    run_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    rdir = os.path.join("output", run_ts)
    fdir = os.path.join(rdir, "figures")
    os.makedirs(fdir, exist_ok=True)
    _log(f"Output folder: {rdir}")
    _log(f"Γ(Q) models: {[_MODEL_LABELS.get(m, m) for m in sel_models]}")
    if method == "bayes":
        _log(f"Bayesian forward model: {_MODEL_LABELS.get(bayes_model, bayes_model)}")

    try:
        #  load
        _prog(8, "Loading data")
        _log("Loading data...")
        staged = _browser.selected_files
        prim   = _browser.primary
        if not staged:
            raise ValueError("No files staged.")
        if not prim:
            raise ValueError("No primary file set.")

        from collections import defaultdict
        by_dir = defaultdict(list)
        for p in staged:
            by_dir[str(pathlib.Path(p).parent)].append(pathlib.Path(p).name)

        dataset = {}
        for dpath, fnames in by_dir.items():
            pn = pathlib.Path(prim).name
            crits = [pn] if str(pathlib.Path(prim).parent) == dpath else []
            dataset.update(load_dataset(fnames, data_dir=dpath,
                                        critical_files=crits, verbose=False))
        if not dataset:
            raise RuntimeError("No files loaded.")

        for d in dataset.values():
            fit_elastic_peak(d)
        # the new assign_resolution accepts a Config but works fine without one
        assign_resolution(dataset, verbose=False)

        pk = pathlib.Path(prim).name
        d_inc = dataset.get(pk, next(iter(dataset.values())))
        if pk not in dataset:
            _log(f"Primary not found in loaded set; using {d_inc['name']}", "warn")
        _log(f"Loaded {len(dataset)} file(s).  Primary: {d_inc['name']}", "ok")

        #  config
        _prog(18, "Configuring")
        cfg = Config(
            files_to_fit  = list(dataset.keys()),
            primary_file  = d_inc["name"],
            q_min         = q_lo,
            q_max         = q_hi,
            n_q_bins      = nbins,
            energy_window = ewin,
            n_walkers     = w_nwalkers.value,
            n_warmup      = w_nwarmup.value,
            n_keep        = w_nkeep.value,
            thin          = 5,
            random_seed   = 42,
            save_dir      = rdir,
        )
        cfg.to_json(os.path.join(rdir, "config.json"))

        #  S(Q,ω)
        _prog(26, "Rendering S(Q,ω) map")
        _log("Rendering S(Q,ω) map...")
        _render(out_map, fig_sqw_map(d_inc, ewin=min(ewin, 1.0)),
                save_path=os.path.join(fdir, "sqw_map.pdf"))

        #  HWHM
        _prog(36, "Extracting Γ(Q)")
        _log("Extracting Γ(Q) per Q-bin...")
        q_hwhm, g_hwhm, g_err, eisf = extract_hwhm(d_inc, cfg)
        if len(q_hwhm) == 0:
            raise RuntimeError("No Q-bins produced a usable HWHM fit.")
        save_hwhm_csv(q_hwhm, g_hwhm, g_err, eisf, rdir)
        _log(f"  {len(q_hwhm)} bins  |  Γ {g_hwhm.min()*1000:.0f}–{g_hwhm.max()*1000:.0f} µeV", "ok")

        #  LS Γ(Q²) fits
        n_m = len(sel_models)
        model_results = {}
        best_model = None
        best_chi2r = np.inf
        best_d, best_l = 0.30, 2.50

        for mi, model in enumerate(sel_models):
            p0 = 40 + mi * (28 // max(n_m, 1))
            mlbl = _MODEL_LABELS.get(model, model)
            _prog(p0, f"LS fit  {mlbl}")
            _log(f"LS Γ(Q²) fit: {mlbl}...")
            ls = _fit_gamma_ls(q_hwhm, g_hwhm, g_err, model)
            model_results[model] = ls

            if "error" in ls:
                _log(f"  fit failed: {ls['error']}", "warn")
                continue

            # reduced χ² vs the data
            try:
                if   model == "ce":      pred = ce_hwhm(q_hwhm, ls["D"], ls["l"])
                elif model == "fickian": pred = fickian_hwhm(q_hwhm, ls["D"])
                elif model == "ss":      pred = ss_hwhm(q_hwhm, ls["D"], ls["tau_s"])
                elif model in _custom_models:
                    info  = _custom_models[model]
                    pvals = [ls.get(pd[0], pd[1]) for pd in info["param_defs"]]
                    pred  = info["func"](q_hwhm, *pvals)
                else:
                    pred = None
                if pred is not None:
                    c2r = float(np.sum(((g_hwhm - pred) / g_err)**2)
                                / max(len(q_hwhm) - len(ls), 1))
                    ls["_chi2r"] = c2r
                    if model == "ce" and c2r < best_chi2r:
                        best_chi2r = c2r
                        best_model = model
                        best_d = ls["D"]
                        best_l = ls["l"]
            except Exception:
                pass
            _log(f"  {ls}", "ok")

        # If CE wasn't picked, fall back to *any* successful fit so we have a
        # spectrum overlay to draw.
        if best_model is None:
            for m, r in model_results.items():
                if "error" not in r and "D" in r:
                    best_d = r["D"]
                    best_l = r.get("l", 2.5)
                    best_model = m
                    break



        #  Bayesian forward-model fit
        samples_full = None
        bayes_summary = None
        if method == "bayes":
            _prog(70, f"Bayesian: {bayes_model}")
            _log(f"Bayesian forward model: {bayes_model}")
            data_bins = build_data_bins(d_inc, cfg)
            qs = np.array([b[3] for b in data_bins])
            # Use a measured kernel from any frozen ref if we have one; else fall
            # back to the scalar Gaussian sigma_res.
            res_ref = None
            for d in dataset.values():
                if d.get("res_source") in ("frozen_sample", "explicit_override") \
                   and d["ei"] == d_inc["ei"]:
                    res_ref = d
                    break
            if res_ref is not None:
                resolution = build_resolution_bins(res_ref, cfg, q_centres=qs)
                _log(f"  using measured resolution kernel from {res_ref['name']}")
            else:
                resolution = d_inc["sigma_res"]
                _log("  using scalar Gaussian resolution σ", "warn")

            try:
                p_map, neg_lnp = find_map(data_bins, resolution,
                                          model=bayes_model, cfg=cfg, verbose=False)
                _log(f"  MAP −lnP={neg_lnp:.2f}  params={p_map}", "ok")
                samples_full = run_mcmc(data_bins, resolution, p_map,
                                        model=bayes_model, cfg=cfg, verbose=False)
                _log(f"  {len(samples_full)} posterior samples", "ok")

                bayes_summary = summarise_samples(
                    samples_full, model=bayes_model, verbose=False,
                    derived=({"D_s/D_t": lambda s: s[:, 3] / np.maximum(s[:, 2], 1e-12)}
                             if bayes_model == "anisotropic_rotor" else None),
                )
                # save raw samples
                np.save(os.path.join(rdir, f"posterior_{bayes_model}.npy"),
                        samples_full)
            except Exception as exc:
                _log(f"  Bayesian fit failed: {exc}", "err")
                samples_full = None



        #  figures
        _prog(86, "Rendering single-Q fit")
        f_fit, chi2r_spec = fig_fit_residuals(d_inc, best_d, best_l, q_target=q_tgt)
        _render(out_fit, f_fit, save_path=os.path.join(fdir, "fit_residuals.pdf"))



        _prog(93, "Rendering Γ(Q²)")
        # If Bayesian samples are 2-D and look like (D, l) we could overlay
        # a posterior fan, but the new forward models are 3-D / 4-D so we
        # leave the Γ(Q²) plot purely-LS.
        _render(out_hwhm,
                fig_hwhm_comparison(q_hwhm, g_hwhm, g_err, model_results,
                                    samples_map=None,
                                    res_hwhm_uev=d_inc["fwhm_res"] / 2 * 1000),
                save_path=os.path.join(fdir, "hwhm_vs_q2.pdf"))

        if samples_full is not None:
            _prog(96, "Rendering posteriors")
            fp = fig_forward_posteriors(samples_full, bayes_model)
            _render(out_post, fp, save_path=os.path.join(fdir, "posteriors.pdf"))
        else:
            with out_post:
                display(HTML('<div class="qs-info" style="margin:16px">'
                             'Posterior plots only appear when "Bayesian MCMC" is enabled.'
                             '</div>'))

        _prog(98, "Compiling results")
        _build_params_table(d_inc, model_results, samples_full, bayes_model,
                            bayes_summary, method, chi2r_spec, q_hwhm, rdir)
        _save_json(d_inc, model_results, samples_full, bayes_model,
                   bayes_summary, method, chi2r_spec, rdir)

        _prog(100, "Complete")
        _log(f"Done. Results saved to: {rdir}", "ok")
        tabs.selected_index = 0

    except Exception as exc:
        with out_params:
            clear_output(wait=True)
            display(HTML(f'<div class="qs-err"><strong>Error:</strong> {exc}'
                         f'<br><pre>{traceback.format_exc()}</pre></div>'))
        _log(f"Error: {exc}", "err")
        w_prog.bar_style = "danger"
        _prog(w_prog.value, "Failed — see Parameters tab")
    finally:
        w_run.disabled = False
        w_run.description = "Run Analysis"


#  results table
def _build_params_table(d_inc, model_results, samples_full, bayes_model,
                        bayes_summary, method, chi2r_spec, q_hwhm, rdir):
    rows = [
        '<tr class="qs-section"><td colspan="4"><strong>Dataset</strong></td></tr>',
        f'<tr><td><strong>File</strong></td><td>{d_inc["name"]}</td><td></td><td></td></tr>',
        f'<tr><td><strong>Temperature</strong></td><td>{d_inc["temp"]} K</td><td></td><td></td></tr>',
        f'<tr><td><strong>Eᵢ</strong></td><td>{d_inc["ei"]:.2f}</td><td>meV</td><td></td></tr>',
        f'<tr><td><strong>Resolution</strong></td><td>{d_inc["fwhm_res"]*1000:.1f}</td><td>µeV FWHM</td><td>{d_inc["res_source"]}</td></tr>',
        f'<tr><td><strong>Q range</strong></td><td>{q_hwhm.min():.3f} – {q_hwhm.max():.3f}</td><td>Å⁻¹</td><td></td></tr>',
        f'<tr><td><strong>Q bins</strong></td><td>{len(q_hwhm)}</td><td></td><td></td></tr>',
        '<tr class="qs-section"><td colspan="4"><strong>Fitting</strong></td></tr>',
        f'<tr><td><strong>Method</strong></td><td>{"Bayesian forward model" if method=="bayes" else "Least squares Γ(Q²)"}</td><td></td><td></td></tr>',
        f'<tr><td><strong>Spectrum χ²ᵣ at Q target</strong></td><td>{chi2r_spec:.3f}</td><td></td><td>ideal ≈ 1</td></tr>',
    ]

    # LS Γ(Q) results
    rows.append('<tr class="qs-section"><td colspan="4"><strong>Least-squares Γ(Q²) fits</strong></td></tr>')
    for model, ls in model_results.items():
        lbl = _MODEL_LABELS.get(model, model)
        rows.append(f'<tr><td colspan="4" style="background:#f8fafc"><strong>{lbl}</strong></td></tr>')
        if "error" in ls:
            rows.append(f'<tr><td colspan="4" style="color:#b91c1c">Fit failed: {ls["error"]}</td></tr>')
            continue
        D = ls.get("D")
        if D is not None:
            rows.append(f'<tr><td><strong>D</strong></td><td>{D:.5f}</td><td>Å²/ps</td><td></td></tr>')
        L = ls.get("l")
        if L is not None:
            rows.append(f'<tr><td><strong>ℓ</strong></td><td>{L:.5f}</td><td>Å</td><td></td></tr>')
            tau = ls.get("tau", L**2 / (6 * D) if D else 0)
            rows.append(f'<tr><td><strong>τ</strong></td><td>{tau:.5f}</td><td>ps</td><td></td></tr>')
        ts = ls.get("tau_s")
        if ts is not None:
            rows.append(f'<tr><td><strong>τₛ</strong></td><td>{ts:.5f}</td><td>ps</td><td></td></tr>')
        mh = ls.get("mean_hwhm_mev")
        if mh is not None:
            rows.append(f'<tr><td><strong>Mean Γ</strong></td><td>{mh*1000:.1f}</td><td>µeV</td><td></td></tr>')
        if model in _custom_models:
            for pd in _custom_models[model]["param_defs"]:
                v = ls.get(pd[0])
                if v is not None:
                    rows.append(f'<tr><td><strong>{pd[0]}</strong></td><td>{v:.5f}</td><td></td><td></td></tr>')
        c2 = ls.get("_chi2r")
        if c2 is not None:
            rows.append(f'<tr><td><strong>χ²ᵣ</strong></td><td>{c2:.4f}</td><td></td><td>ideal ≈ 1</td></tr>')

    # Bayesian forward-model results
    if samples_full is not None and bayes_summary is not None:
        rows.append('<tr class="qs-section"><td colspan="4"><strong>Bayesian forward model</strong></td></tr>')
        rows.append(f'<tr><td><strong>Model</strong></td><td>{_MODEL_LABELS.get(bayes_model, bayes_model)}</td><td></td><td>{len(samples_full)} samples</td></tr>')
        for name, (med, lo, hi) in bayes_summary.items():
            rows.append(f'<tr><td><strong>{name}</strong></td><td>{med:.5f}</td><td></td>'
                        f'<td>95% CI [{lo:.5f}, {hi:.5f}]</td></tr>')

    table = ('<table class="qs-tbl"><thead><tr><th>Parameter</th><th>Value</th>'
             '<th>Unit</th><th>Notes</th></tr></thead><tbody>'
             + "".join(rows) + '</tbody></table>')
    banner = (f'<div class="qs-ok" style="margin-bottom:12px">'
              f'Analysis complete — results saved to <code>{rdir}</code></div>')
    with out_params:
        clear_output(wait=True)
        display(HTML(banner + table))


def _save_json(d_inc, model_results, samples_full, bayes_model, bayes_summary,
               method, chi2r_spec, rdir):
    out = {
        "timestamp":      datetime.datetime.now().isoformat(),
        "qens_version":   qens.__version__,
        "dataset":        d_inc["name"],
        "temperature_K":  int(d_inc["temp"]),
        "Ei_meV":         float(d_inc["ei"]),
        "method":         method,
        "res_fwhm_meV":   float(d_inc["fwhm_res"]),
        "res_source":     d_inc["res_source"],
        "spectrum_chi2r": float(chi2r_spec),
        "ls_models":      {},
    }
    for model, ls in model_results.items():
        entry = {k: float(v) for k, v in ls.items()
                 if k != "error" and isinstance(v, (int, float, np.floating))}
        out["ls_models"][model] = entry

    if samples_full is not None and bayes_summary is not None:
        out["bayesian"] = {
            "model": bayes_model,
            "n_samples": int(len(samples_full)),
            "summary": {n: {"median": m, "ci95_lo": lo, "ci95_hi": hi}
                        for n, (m, lo, hi) in bayes_summary.items()},
        }

    with open(os.path.join(rdir, "results.json"), "w") as fh:
        json.dump(out, fh, indent=2)


# --------------------------------------------------------- preview from file browser
def _preview_primary():
    """Render a quick preview of the selected primary file without running a full fit."""
    prim = _browser.primary
    if not prim or not os.path.exists(prim):
        return
    with out_map:
        clear_output(wait=True)
        display(HTML('<div class="qs-info" style="margin:16px 0">Loading preview…</div>'))
    try:
        d = read_nxspe(prim)
        fit_elastic_peak(d)
        assign_resolution({d["name"]: d}, verbose=False)
        fig = fig_data_preview(d)
        _render(out_map, fig)
        tabs.selected_index = 1
        _log(f"Preview: {d['name']}  Eᵢ={d['ei']:.2f} meV  "
             f"T={d['temp']} K  good={len(d['good'])} det", "ok")
    except Exception as exc:
        with out_map:
            clear_output(wait=True)
            display(HTML(f'<div class="qs-err"><strong>Preview failed:</strong> {exc}'
                         f'<br><span style="font-size:.82em">'
                         f'Call <code>inspect_nxspe(path)</code> to check file structure.'
                         f'</span></div>'))
        _log(f"Preview error: {exc}", "err")


_browser._preview_cb = _preview_primary
w_run.on_click(run_pipeline)
print("Pipeline wired. Ready.")


Pipeline wired. Ready.


## 9. Launch the app

Run this final cell after all previous cells have completed. It displays the interface and a short reminder about the file browser.

In [9]:
# Display the finished application.
# Re-run this cell after editing layout/styling cells above to refresh the interface.

display(HTML(_CSS))
display(app)
display(HTML('<div class="qs-info" style="margin-top:10px;max-width:940px;font-size:.85em">'
             '<strong>File browser:</strong> &nbsp;Click a folder to navigate into it &middot; '
             '<em>Up</em> to go back &middot; Ctrl-click / Shift-click in the file list for multi-select &middot; '
             '<em>Add selected</em> stages highlighted files &middot; <em>Add all</em> stages every .nxspe in the current folder &middot; '
             'Set the <em>Primary file</em> dropdown to your INC reference.'
             '</div>'))